# Phase A — Train RQ-VAE trên Kaggle

Notebook clone repository, tự tìm embedding từ notebook 01 rồi chạy `train_rqvae.py`. Hyperparameter lấy từ `configs/rqvae_kuaisearch.gin`; notebook chỉ thay đường dẫn input.

Đầu vào bắt buộc từ notebook 01:

- `global_product_embeddings.f16.npy`
- `global_embedding_index.parquet`

Đầu ra của bước này nằm trong `/kaggle/working/rq-vae`:

- checkpoint RQ-VAE
- `semantic_ids.parquet` chứa ba tầng SID và collision suffix deterministic

RQ-VAE dùng ba codebook giảm dần `128 × 64 × 32`. Sau khi train, notebook freeze mapping và thêm collision suffix để full SID xác định duy nhất một sản phẩm.

## Trước khi chạy

1. Bật GPU trong Kaggle Notebook Settings.
2. Add Data chứa output của notebook 01.
3. Tạo Kaggle Secret `GITHUB_TOKEN` có quyền đọc repository.
4. Tạo Kaggle Secret `WANDB_API_KEY`.
5. Notebook tự tìm embedding trong `/kaggle/input`; các cấu hình train còn lại lấy từ Gin.

## 0. Cấu hình

In [1]:
from pathlib import Path

PREPROCESSED_ROOT = None

GITHUB_REPOSITORY_URL = "https://github.com/nam-htran/VSF-MiniApp-Ecommerce.git"
GITHUB_BRANCH = "main"
REPOSITORY_ROOT = Path("/kaggle/working/vsf-miniapp-ecommerce-source")
AUTO_INSTALL_DEPENDENCIES = True

print("Configuration loaded.")

Configuration loaded.


## 1. Cài dependency và kiểm tra GPU

In [2]:
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys


required_modules = {
    "gin": "gin-config==0.5.0",
    "accelerate": "accelerate>=1.0.0",
    "einops": "einops>=0.8.0",
    "huggingface_hub": "huggingface-hub>=0.25.0",
    "wandb": "wandb>=0.19.0",
    "pyarrow": "pyarrow>=16.0.0",
}
missing_packages = [
    package
    for module, package in required_modules.items()
    if importlib.util.find_spec(module) is None
]
if AUTO_INSTALL_DEPENDENCIES and missing_packages:
    print("Installing:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {properties.name}, {properties.total_memory / 2**30:.1f} GiB")

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training RQ-VAE.")

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
cuda:0: Tesla T4, 14.6 GiB
cuda:1: Tesla T4, 14.6 GiB


## 2. Kết nối Weights & Biases

Đọc `WANDB_API_KEY` từ Kaggle Secret và đăng nhập W&B.

In [3]:
import os


from kaggle_secrets import UserSecretsClient

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## 3. Clone source từ GitHub

Đọc `GITHUB_TOKEN` từ Kaggle Secret và clone nhánh `main`.

In [4]:
from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPOSITORY_ROOT = Path(REPOSITORY_ROOT).expanduser().resolve()

git_environment = {
    **os.environ,
    "GITHUB_TOKEN": github_token,
    "GIT_TERMINAL_PROMPT": "0",
}
credential_helper = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
git = ["git", "-c", f"credential.helper={credential_helper}"]

if (REPOSITORY_ROOT / ".git").is_dir():
    subprocess.run(
        [*git, "-C", str(REPOSITORY_ROOT), "pull", "--ff-only", "origin", GITHUB_BRANCH],
        check=True,
        env=git_environment,
    )
elif REPOSITORY_ROOT.exists():
    raise FileExistsError(f"Clone target is not a Git repository: {REPOSITORY_ROOT}")
else:
    subprocess.run(
        [*git, "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPOSITORY_URL, str(REPOSITORY_ROOT)],
        check=True,
        env=git_environment,
    )
del github_token, git_environment

SOURCE_ROOT = REPOSITORY_ROOT / "pi-recommendation/src"
if not (SOURCE_ROOT / "train_rqvae.py").is_file():
    raise FileNotFoundError(f"RQ-VAE source not found: {SOURCE_ROOT}")
print("SOURCE_ROOT:", SOURCE_ROOT)

Cloning into '/kaggle/working/vsf-miniapp-ecommerce-source'...


SOURCE_ROOT: /kaggle/working/vsf-miniapp-ecommerce-source/pi-recommendation/src


## 4. Tìm output notebook 01 và tạo Gin runtime

In [5]:
import json


def is_preprocessed_root(path):
    path = Path(path)
    return (
        (path / "global_product_embeddings.f16.npy").is_file()
        and (path / "global_embedding_index.parquet").is_file()
    )


def locate_preprocessed_root(explicit=None):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if is_preprocessed_root(root):
            return root
        raise FileNotFoundError(f"Notebook 01 artifacts were not found in {root}")

    cwd = Path.cwd().resolve()
    candidates = [
        Path("/kaggle/working/preprocessed"),
        cwd / "preprocessed",
        cwd.parent / "preprocessed",
    ]
    for candidate in candidates:
        if is_preprocessed_root(candidate):
            return candidate.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for manifest_path in kaggle_input.glob("**/preprocessing_manifest.json"):
            if is_preprocessed_root(manifest_path.parent):
                return manifest_path.parent.resolve()
        for embedding_path in kaggle_input.glob("**/global_product_embeddings.f16.npy"):
            if is_preprocessed_root(embedding_path.parent):
                return embedding_path.parent.resolve()
    raise FileNotFoundError(
        "Notebook 01 output was not found. Add it as a Kaggle Dataset or set PREPROCESSED_ROOT."
    )


PREPROCESSED_ROOT = locate_preprocessed_root(PREPROCESSED_ROOT)

BASE_CONFIG_PATH = SOURCE_ROOT / "configs/rqvae_kuaisearch.gin"
CONFIG_PATH = Path("/kaggle/working/rqvae_kuaisearch.gin")
config_lines = BASE_CONFIG_PATH.read_text(encoding="utf-8").splitlines()
binding = "train.dataset_folder="
matching_lines = [index for index, line in enumerate(config_lines) if line.startswith(binding)]
if len(matching_lines) != 1:
    raise ValueError(f"Expected one {binding} binding, found {len(matching_lines)}")
config_lines[matching_lines[0]] = binding + json.dumps(str(PREPROCESSED_ROOT))
CONFIG_PATH.write_text("\n".join(config_lines) + "\n", encoding="utf-8")

print("Preprocessed root:", PREPROCESSED_ROOT)
print("Gin config:", CONFIG_PATH)

Preprocessed root: /kaggle/input/notebooks/yennhung1/vsf-pi-01/preprocessed
Gin config: /kaggle/working/rqvae_kuaisearch.gin


## 5. Train RQ-VAE

In [6]:
command = [sys.executable, "train_rqvae.py", str(CONFIG_PATH)]
print("Running:", " ".join(command))
subprocess.run(command, cwd=SOURCE_ROOT, check=True)

Running: /usr/bin/python3 train_rqvae.py /kaggle/working/rqvae_kuaisearch.gin


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hnamt04 (hnamt04-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/vsf-miniapp-ecommerce-source/pi-recommendation/src/wandb/run-20260816_034943-if627hy1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run rotation-128x64x32-50k-20260816-034943
wandb: ⭐️ View project at https://wandb.ai/hnamt04-personal/kuaisearch-rqvae
wandb: 🚀 View run at https://wandb.ai/hnamt04-personal/kuaisearch-rqvae/runs/if627hy1


Device: cuda


loss: 0.0459, rl: 0.0439, vl: 0.0020: 100%|██████████| 50000/50000 [13:29<00:00, 61.77it/s]
wandb: updating run metadata


Semantic IDs saved to /kaggle/working/rq-vae/semantic_ids.parquet


CompletedProcess(args=['/usr/bin/python3', 'train_rqvae.py', '/kaggle/working/rqvae_kuaisearch.gin'], returncode=0)

## 6. Kiểm tra artifacts

In [7]:
OUTPUT_ROOT = Path("/kaggle/working/rq-vae")
checkpoints = list(OUTPUT_ROOT.glob("checkpoint_*.pt"))
checkpoints.sort(key=lambda path: int(path.stem.rsplit("_", 1)[1]))

In [8]:
import pandas as pd


SEMANTIC_IDS_PATH = OUTPUT_ROOT / "semantic_ids.parquet"

if not checkpoints:
    raise FileNotFoundError("RQ-VAE finished without a checkpoint.")
if not SEMANTIC_IDS_PATH.is_file():
    raise FileNotFoundError("RQ-VAE finished without semantic_ids.parquet.")

semantic_ids = pd.read_parquet(SEMANTIC_IDS_PATH)
full_sid_columns = ["sid_0", "sid_1", "sid_2", "sid_suffix"]
if not set(full_sid_columns).issubset(semantic_ids.columns):
    raise ValueError(f"Missing Semantic ID columns: {full_sid_columns}")
if semantic_ids.duplicated(full_sid_columns).any():
    raise ValueError("Full Semantic IDs must be unique.")

output_size = sum(path.stat().st_size for path in OUTPUT_ROOT.rglob("*") if path.is_file())

print("Artifact validation: PASSED")
print("Checkpoints:", len(checkpoints))
print("Latest checkpoint:", checkpoints[-1])
print("Semantic IDs:", SEMANTIC_IDS_PATH)
print("Maximum collision suffix:", int(semantic_ids["sid_suffix"].max()))
print("Output size:", f"{output_size / 2**30:.2f} GiB")
print("OUTPUT_ROOT:", OUTPUT_ROOT)

Artifact validation: PASSED
Checkpoints: 10
Latest checkpoint: /kaggle/working/rq-vae/checkpoint_49999.pt
Semantic IDs: /kaggle/working/rq-vae/semantic_ids.parquet
Maximum collision suffix: 7308
Output size: 0.11 GiB
OUTPUT_ROOT: /kaggle/working/rq-vae


## Khi nào notebook hoàn thành?

Notebook hoàn thành khi cell cuối báo `Artifact validation: PASSED`. Lúc đó checkpoint và `semantic_ids.parquet` đều đã được tạo trong `/kaggle/working/rq-vae`.